# Test: matching de geometria contra llegenda (pipeline real del backend)

Aquest notebook reprodueix exactament la lògica de `POST /api/upload` del backend
(`app/services/dxf_service.py`, `app/services/legend_matching_service.py`),
aplicada a un fitxer `plan.dxf` de Google Drive:

1. Agrupa la geometria mesurable de `plan.dxf` per estil `(colorHex, linetype, lineweight)`.
2. Enganxa aquí el JSON de llegenda obtingut al notebook `Test-Legend-Reading.ipynb`
   (llista d'objectes `key/label/colorHex/linetype/lineweight`).
3. Fa el matching: el linetype ha de coincidir exactament; el color ha
   d'estar dins la tolerància `color_match_tolerance` (30.0, distància
   euclidiana RGB), tal com fa `match_geometry_to_legend`. El lineweight
   **no** és un criteri de matching -- es guarda com a dada informativa,
   però no bloqueja un match perquè en DXFs reals sovint es resol de
   manera diferent entre la llegenda i la geometria del mateix material.
4. Mostra els materials **coincidents** (amb metres lineals -- l'objectiu
   real de l'app) i la geometria **indeterminada** (que no ha trobat cap
   fila de llegenda compatible).


In [ ]:
# %% [Cel·la 1] Instal·lar dependències
!pip install -q ezdxf pandas

In [ ]:
# %% [Cel·la 2] Muntar Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# %% [Cel·la 3] Ruta al fitxer plan.dxf a Drive
DXF_PATH = "/content/drive/MyDrive/Colab Notebooks/plan-count/PB_sola.dxf"
print(DXF_PATH)

In [ ]:
# %% [Cel·la 4] Obrir el fitxer
import ezdxf

doc = ezdxf.readfile(DXF_PATH)
msp = doc.modelspace()
print("DXF llegit correctament. Entitats a l'espai model:", len(msp))

In [ ]:
# %% [Cel·la 5] Còpia exacta de app/services/dxf_service.py (les mateixes
# funcions que fa servir el notebook de llegenda, aplicades ara a plan.dxf).
import ezdxf.colors as ezcolors
import ezdxf.lldxf.const


def _safe_layer(doc, name):
    try:
        return doc.layers.get(name)
    except ezdxf.lldxf.const.DXFTableEntryError:
        return None


def iter_with_blocks(entities, depth=0, max_depth=3):
    for entity in entities:
        yield entity
        if entity.dxftype() == "INSERT" and depth < max_depth:
            try:
                nested = list(entity.virtual_entities())
            except Exception:
                nested = []
            yield from iter_with_blocks(nested, depth + 1, max_depth)


def _effective_color_hex(entity, doc) -> str:
    true_color = entity.dxf.get("true_color", None)
    if true_color is not None:
        r, g, b = ezcolors.int2rgb(true_color)
        return "#%02x%02x%02x" % (r, g, b)
    aci = entity.dxf.color
    if aci == 256:
        layer = _safe_layer(doc, entity.dxf.layer)
        aci = layer.color if layer else 7
    elif aci == 0:
        aci = 7
    return "#%02x%02x%02x" % ezcolors.aci2rgb(aci)


def _effective_linetype(entity, doc) -> str:
    linetype = entity.dxf.linetype
    if linetype == "BYLAYER":
        layer = _safe_layer(doc, entity.dxf.layer)
        linetype = layer.dxf.linetype if layer else "CONTINUOUS"
    return linetype.upper()


def _effective_lineweight(entity, doc) -> int:
    lineweight = entity.dxf.lineweight
    if lineweight == -1:
        layer = _safe_layer(doc, entity.dxf.layer)
        return layer.dxf.lineweight if layer else -3
    return lineweight


def _entity_length(entity):
    dxftype = entity.dxftype()
    if dxftype == "LINE":
        s, e = entity.dxf.start, entity.dxf.end
        return ((s[0] - e[0]) ** 2 + (s[1] - e[1]) ** 2) ** 0.5
    if dxftype == "LWPOLYLINE":
        points = list(entity.get_points("xy"))
        if len(points) < 2:
            return None
        total = 0.0
        for i in range(len(points) - 1):
            a, b = points[i], points[i + 1]
            total += ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5
        if entity.closed:
            a, b = points[-1], points[0]
            total += ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5
        return total
    return None


def group_measurable_geometry_by_style(doc):
    msp = doc.modelspace()
    totals = {}
    for entity in iter_with_blocks(msp):
        length = _entity_length(entity)
        if length is None:
            continue
        style = (
            _effective_color_hex(entity, doc),
            _effective_linetype(entity, doc),
            _effective_lineweight(entity, doc),
        )
        totals[style] = totals.get(style, 0.0) + length
    return {s: t for s, t in totals.items() if t != 0.0}


style_groups = group_measurable_geometry_by_style(doc)
print(f"Grups d'estil trobats a plan.dxf: {len(style_groups)}")
for style, total in style_groups.items():
    print(style, round(total, 3))

In [ ]:
# %% [Cel·la 5b] Còpia exacta de detect_unit() a app/services/dxf_service.py --
# detecta la unitat real de plan.dxf a partir de la variable de capçalera
# $INSUNITS. És el camp nou `detectedUnit` que ara torna `/api/upload`.
# Si no es reconeix (unitless, imperial, valor no suportat) es fa fallback
# a metres, que és el que aquesta app ha assumit sempre implícitament.
import ezdxf.units as ezunits

_SUPPORTED_UNITS = {"mm", "cm", "m"}


def detect_unit(doc) -> str:
    raw = doc.header.get("$INSUNITS", 0)
    name = ezunits.decode(raw)
    return name if name in _SUPPORTED_UNITS else "m"


detected_unit = detect_unit(doc)
print(f"$INSUNITS de plan.dxf: {doc.header.get('$INSUNITS', 0)} -> detectedUnit: {detected_unit!r}")

In [ ]:
# %% [Cel·la 6] Enganxa aquí el JSON de llegenda copiat del notebook
# Test-Legend-Reading.ipynb (llista d'objectes key/label/colorHex/linetype/lineweight/isDashed).
import json

LEGEND_JSON = """
[
  {
    "key": "R3",
    "label": "TRASDOSSAT AUTOPÒRTANT (6,1 cm)",
    "colorHex": "#2776bb",
    "linetype": "CONTINUOUS",
    "lineweight": 0,
    "isDashed": false
  },
  {
    "key": "R3*",
    "label": "TRASDOSSAT AUTOPÒRTANT (6,1 cm)",
    "colorHex": "#2776bb",
    "linetype": "LÍNEAS_OCULTAS2",
    "lineweight": 0,
    "isDashed": true
  },
  {
    "key": "R4",
    "label": "ENVÀ GUIX LAMINAT (10 cm)",
    "colorHex": "#991b1e",
    "linetype": "CONTINUOUS",
    "lineweight": 0,
    "isDashed": false
  },
  {
    "key": "R4*",
    "label": "ENVÀ GUIX LAMINAT (10 cm)",
    "colorHex": "#991b1e",
    "linetype": "LÍNEAS_OCULTAS2",
    "lineweight": 0,
    "isDashed": true
  },
  {
    "key": "R6",
    "label": "TRASDOSSAT PILARS (3 cm)",
    "colorHex": "#93278f",
    "linetype": "CONTINUOUS",
    "lineweight": 0,
    "isDashed": false
  },
  {
    "key": "R1",
    "label": "FAÇANA SATE (32 cm)",
    "colorHex": "#f8991e",
    "linetype": "CONTINUOUS",
    "lineweight": 0,
    "isDashed": false
  },
  {
    "key": "R2",
    "label": "FAÇANA SATE + PORCELÀNIC (32 cm)",
    "colorHex": "#f8991e",
    "linetype": "LÍNEAS_OCULTAS2",
    "lineweight": 0,
    "isDashed": true
  },
  {
    "key": "R7",
    "label": "Arrebossat amb morter i pintat",
    "colorHex": "#f8b08b",
    "linetype": "LÍNEAS_OCULTAS2",
    "lineweight": 0,
    "isDashed": true
  },
  {
    "key": "R5",
    "label": "PARET D'OBRA DE FÀBRICA (15 cm)",
    "colorHex": "#139b48",
    "linetype": "CONTINUOUS",
    "lineweight": 0,
    "isDashed": false
  }
]
"""

legend_entries = json.loads(LEGEND_JSON)
print(f"Files de llegenda carregades: {len(legend_entries)}")
for e in legend_entries:
    print(e)

In [ ]:
# %% [Cel·la 7] Còpia exacta de app/services/legend_matching_service.py


def _hex_to_rgb(color_hex: str):
    color_hex = color_hex.lstrip("#")
    return tuple(int(color_hex[i : i + 2], 16) for i in (0, 2, 4))


def _rgb_distance(a, b) -> float:
    return sum((x - y) ** 2 for x, y in zip(a, b)) ** 0.5


def _find_matching_entry(color_hex, linetype, lineweight, legend, color_tolerance):
    # lineweight no és un criteri de matching: en DXFs reals sovint es
    # resol de manera diferent entre la fila de llegenda i la geometria del
    # mateix material (p.ex. BYLAYER en capes diferents), i exigir-ne la
    # igualtat exacta descartava matches correctes cap a "undetermined".
    rgb = _hex_to_rgb(color_hex)
    best_entry = None
    best_distance = None
    for entry in legend:
        if entry["linetype"].upper() != linetype.upper():
            continue
        distance = _rgb_distance(rgb, _hex_to_rgb(entry["colorHex"]))
        if distance <= color_tolerance and (best_distance is None or distance < best_distance):
            best_entry, best_distance = entry, distance
    return best_entry


def match_geometry_to_legend(style_groups, legend, color_tolerance=30.0):
    matched_totals = {}
    matched_labels = {}
    matched_colors = {}
    matched_is_dashed = {}
    undetermined = []

    for (color_hex, linetype, lineweight), total in style_groups.items():
        entry = _find_matching_entry(color_hex, linetype, lineweight, legend, color_tolerance)
        if entry is None:
            undetermined.append(
                {"colorHex": color_hex, "linetype": linetype, "lineweight": lineweight, "linearMeters": round(total, 3)}
            )
            continue
        key = entry["key"]
        matched_totals[key] = matched_totals.get(key, 0.0) + total
        matched_labels[key] = entry["label"]
        matched_colors[key] = entry["colorHex"]
        matched_is_dashed[key] = entry["isDashed"]

    matched = [
        {
            "key": key,
            "label": matched_labels[key],
            "colorHex": matched_colors[key],
            "isDashed": matched_is_dashed[key],
            "linearMeters": round(total, 3),
        }
        for key, total in matched_totals.items()
    ]
    return matched, undetermined


COLOR_MATCH_TOLERANCE = 30.0  # ha de coincidir amb `color_match_tolerance` a app/config.py

matched, undetermined = match_geometry_to_legend(style_groups, legend_entries, COLOR_MATCH_TOLERANCE)
print(f"Materials coincidents: {len(matched)}")
print(f"Grups indeterminats: {len(undetermined)}")

In [ ]:
# %% [Cel·la 8] Mostrar resultats -- equivalent al JSON de resposta de `/api/upload`
# (ara amb el camp `detectedUnit`: matched/undetermined segueixen en unitats
# DXF crudes -- el backend no les escala -- és el frontend qui les multiplica
# pel factor de `detectedUnit` (mm=0.001, cm=0.01, m=1) abans de mostrar-les).
# `isDashed` a `matched` és el que permet al frontend pintar el swatch de
# cada material com a barra sòlida o discontínua, sense necessitat de
# mostrar el lineweight a l'usuari.
import json

import pandas as pd

print("--- Materials coincidents (matched) ---")
matched_df = pd.DataFrame(matched, columns=["key", "label", "colorHex", "isDashed", "linearMeters"])
display(matched_df)

print("\n--- Geometria indeterminada (undetermined) ---")
undetermined_df = pd.DataFrame(undetermined, columns=["colorHex", "linetype", "lineweight", "linearMeters"])
display(undetermined_df)

print(f"\n--- detectedUnit: {detected_unit!r} ---")

upload_response = {"matched": matched, "undetermined": undetermined, "detectedUnit": detected_unit}
print("\n--- JSON equivalent a la resposta de /api/upload ---\n")
print(json.dumps(upload_response, ensure_ascii=False, indent=2))